# Notebook 2 — Bayesian Hyperparameter Tuning

Ottimizzazione bayesiana degli iperparametri di **ResiDual CLAP** con:
- `n_components_ratio` — frazione di componenti PCA da usare
- `target_layers` — quali layer del transformer applicare il reweighting

**Strategia:**
- Gaussian Process Regression come surrogate model (scikit-optimize)
- Acquisition function: Expected Improvement
- Punto di partenza: valutazione baseline CLAP (calcolata una volta sola)
- Ogni trial: fit PCA + valutazione su campione fisso

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES (solo prima volta su Kaggle)
# ============================================================
!pip install scikit-optimize --quiet

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import sys
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, '/kaggle/input/clap-project')  # Adatta al tuo dataset Kaggle

from residual_clap_utils import (
    BayesianHPTuner,
    build_clap,
    evaluate_model,
    accuracy,
    get_dataset_registry,
    plot_hp_results,
    print_hp_table,
)

print("✅ Import completati")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# ============================================================
# CONFIGURAZIONE GLOBALE
# ============================================================

# --- Dataset target ---
DATASET_NAME = 'VocalSound'           # Cambia col dataset che vuoi usare
DATASET_ROOT = '/kaggle/input/vocalsound'
PROMPT       = 'this is the sound of '
CLAP_VERSION = '2023'
RANDOM_STATE = 42

# --- Parametri ottimizzazione ---
N_CALLS           = 25    # Numero totale di trial (inclusi initial_points)
N_INITIAL_POINTS  = 6     # Trial random iniziali (exploration)
EVAL_SAMPLES      = 200   # Campioni per valutare ogni configurazione
PCA_SAMPLES       = 300   # Campioni per fitting PCA

# --- Spazio di ricerca ---
PARAM_SPACE = {
    # Intervallo per n_components_ratio
    'n_components_ratio': (0.05, 0.50),
    
    # Tutte le combinazioni di target_layers da esplorare
    # Swin Transformer HTSAT ha 4 layer (0, 1, 2, 3)
    'target_layers_options': [
        [0],
        [1],
        [2],
        [3],
        [0, 1],
        [1, 2],
        [2, 3],
        [0, 1, 2],
        [1, 2, 3],
        [0, 1, 2, 3],
    ],
}

# --- Config base (mode e altri parametri fissi) ---
BASE_RESIDUAL_CONFIG = {
    'mode': 'attention',
    'whitening_strength': 1.0,
    'whitening_eps': 1e-6,
    # n_components_ratio e target_layers saranno sovrascritti dall'optimizer
    'n_components_ratio': 0.15,
    'target_layers': [3],
}

print("Configurazione HP Tuning:")
print(f"  Dataset:          {DATASET_NAME}")
print(f"  N_CALLS:          {N_CALLS}")
print(f"  N_INITIAL_POINTS: {N_INITIAL_POINTS}")
print(f"  EVAL_SAMPLES:     {EVAL_SAMPLES}")
print(f"  PCA_SAMPLES:      {PCA_SAMPLES}")
print(f"  n_comp_ratio:     {PARAM_SPACE['n_components_ratio']}")
print(f"  n_layer_combos:   {len(PARAM_SPACE['target_layers_options'])}")

In [ ]:
# ============================================================
# CARICAMENTO DATASET
# ============================================================

DATASET_REGISTRY = get_dataset_registry()

if DATASET_NAME not in DATASET_REGISTRY:
    raise ValueError(f"Dataset '{DATASET_NAME}' non trovato. Disponibili: {list(DATASET_REGISTRY.keys())}")

print(f"Caricamento {DATASET_NAME}...")
dataset = DATASET_REGISTRY[DATASET_NAME](root=DATASET_ROOT, download=False)
text_labels = [PROMPT + x for x in dataset.classes]

print(f"✅ {DATASET_NAME}: {len(dataset)} campioni, {len(dataset.classes)} classi")
print(f"   Classi: {dataset.classes}")
print(f"   Prompt labels: {text_labels[:3]} ...")

In [ ]:
# ============================================================
# CALCOLO BASELINE (una volta sola, punto di riferimento)
# ============================================================

print("\nCalcolo accuracy baseline (CLAP standard)...")

clap_std = build_clap(
    model_type='classic',
    version=CLAP_VERSION,
    use_cuda=torch.cuda.is_available()
)

rng = np.random.RandomState(RANDOM_STATE)
baseline_indices = rng.choice(
    len(dataset),
    size=min(EVAL_SAMPLES, len(dataset)),
    replace=False
).tolist()

y_preds_b, y_labels_b = evaluate_model(
    wrapper=clap_std,
    dataset=dataset,
    text_labels=text_labels,
    indices=baseline_indices,
    desc="Baseline CLAP",
)
baseline_acc = accuracy(y_preds_b, y_labels_b)

del clap_std
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n✅ Baseline accuracy: {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")

In [ ]:
# ============================================================
# ESECUZIONE BAYESIAN OPTIMIZATION
# ============================================================

start_time = datetime.now()
print(f"\nInizio ottimizzazione: {start_time.strftime('%H:%M:%S')}")

tuner = BayesianHPTuner(
    dataset=dataset,
    text_labels=text_labels,
    base_residual_config=BASE_RESIDUAL_CONFIG,
    param_space=PARAM_SPACE,
    n_calls=N_CALLS,
    n_initial_points=N_INITIAL_POINTS,
    eval_samples=EVAL_SAMPLES,
    pca_samples=PCA_SAMPLES,
    clap_version=CLAP_VERSION,
    use_cuda=torch.cuda.is_available(),
    random_state=RANDOM_STATE,
)

# Iniettiamo manualmente il baseline già calcolato per non ricalcolarlo
tuner._baseline_acc = baseline_acc

best_params, best_score, all_results = tuner.run(verbose=True)

end_time = datetime.now()
elapsed = (end_time - start_time).total_seconds() / 60
print(f"\n⏱️  Ottimizzazione completata in {elapsed:.1f} minuti")

In [ ]:
# ============================================================
# TABELLA TOP CONFIGURAZIONI
# ============================================================

print_hp_table(all_results, top_n=10)

In [ ]:
# ============================================================
# VISUALIZZAZIONE RISULTATI
# ============================================================

fig = plot_hp_results(all_results, baseline_acc, figsize=(16, 5))
plt.suptitle(
    f'Bayesian HP Tuning — {DATASET_NAME}\n'
    f'Baseline: {baseline_acc:.3f}  |  Best: {best_score:.3f}  '
    f'(Δ={best_score - baseline_acc:+.3f})',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('/kaggle/working/hp_tuning_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Plot salvato in hp_tuning_overview.png")

In [ ]:
# ============================================================
# HEATMAP n_components_ratio × target_layers
# ============================================================

from collections import defaultdict

# Aggrega accuracy media per ogni combo (n_comp_bin × target_layers)
N_BINS = 5
comp_bins = np.linspace(
    PARAM_SPACE['n_components_ratio'][0],
    PARAM_SPACE['n_components_ratio'][1],
    N_BINS + 1
)
comp_labels = [f"{comp_bins[i]:.2f}-{comp_bins[i+1]:.2f}" for i in range(N_BINS)]

layers_options_str = [str(opt) for opt in PARAM_SPACE['target_layers_options']]

heatmap_data = np.full((N_BINS, len(layers_options_str)), np.nan)
count_data   = np.zeros((N_BINS, len(layers_options_str)), dtype=int)

for r in all_results:
    comp = r['n_components_ratio']
    lstr = str(r['target_layers'])
    
    bin_idx = min(np.searchsorted(comp_bins[1:], comp), N_BINS - 1)
    
    if lstr in layers_options_str:
        l_idx = layers_options_str.index(lstr)
        if np.isnan(heatmap_data[bin_idx, l_idx]):
            heatmap_data[bin_idx, l_idx] = r['accuracy']
            count_data[bin_idx, l_idx] = 1
        else:
            heatmap_data[bin_idx, l_idx] += r['accuracy']
            count_data[bin_idx, l_idx] += 1

# Media
mask = count_data > 0
heatmap_avg = np.where(mask, heatmap_data / np.where(mask, count_data, 1), np.nan)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap_avg, aspect='auto', cmap='RdYlGn',
               vmin=max(0, baseline_acc - 0.1),
               vmax=min(1, baseline_acc + 0.15))

ax.set_xticks(range(len(layers_options_str)))
ax.set_xticklabels(layers_options_str, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(N_BINS))
ax.set_yticklabels(comp_labels, fontsize=9)
ax.set_xlabel('target_layers')
ax.set_ylabel('n_components_ratio')
ax.set_title(f'Heatmap accuracy media — {DATASET_NAME}\n'
             f'(baseline = {baseline_acc:.3f}, celle grigie = nessun trial)')

plt.colorbar(im, ax=ax, label='Accuracy')

# Annotazioni
for i in range(N_BINS):
    for j in range(len(layers_options_str)):
        if not np.isnan(heatmap_avg[i, j]):
            ax.text(j, i, f'{heatmap_avg[i,j]:.2f}\n(n={count_data[i,j]})',
                    ha='center', va='center', fontsize=7,
                    color='black' if abs(heatmap_avg[i,j] - baseline_acc) < 0.05 else 'white')

plt.tight_layout()
plt.savefig('/kaggle/working/hp_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Heatmap salvata in hp_heatmap.png")

In [ ]:
# ============================================================
# CONVERGENCE PLOT (regret cumulativo)
# ============================================================

sorted_by_trial = sorted(all_results, key=lambda x: x['trial'])
accs_by_trial = [r['accuracy'] for r in sorted_by_trial]
best_so_far   = [max(accs_by_trial[:i+1]) for i in range(len(accs_by_trial))]
regret        = [best_so_far[-1] - b for b in best_so_far]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(range(1, len(accs_by_trial) + 1), accs_by_trial, 'o-', alpha=0.5,
        color='coral', markersize=4, label='Trial accuracy')
ax.plot(range(1, len(best_so_far) + 1), best_so_far, 'g-', linewidth=2,
        label='Best so far')
ax.axhline(baseline_acc, color='steelblue', linestyle='--',
           label=f'Baseline ({baseline_acc:.3f})')
ax.axvline(N_INITIAL_POINTS, color='gray', linestyle=':', alpha=0.7,
           label=f'End random phase ({N_INITIAL_POINTS})')
ax.set_xlabel('Trial')
ax.set_ylabel('Accuracy')
ax.set_title('Convergenza ottimizzazione')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(range(1, len(regret) + 1), regret, 'o-', color='purple', alpha=0.7)
ax2.axvline(N_INITIAL_POINTS, color='gray', linestyle=':', alpha=0.7)
ax2.set_xlabel('Trial')
ax2.set_ylabel('Gap dal best finale')
ax2.set_title('Regret cumulativo')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/hp_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VALIDAZIONE CONFIGURAZIONE OTTIMALE
# ============================================================

print(f"\n{'='*60}")
print("VALIDAZIONE CONFIGURAZIONE OTTIMALE")
print(f"{'='*60}")
print(f"  n_components_ratio: {best_params['n_components_ratio']:.4f}")
print(f"  target_layers:      {best_params['target_layers']}")
print(f"\nEsecuzione validazione su campione indipendente...")

# Usa indici diversi da quelli usati nei trial
all_trial_indices = set(
    list(np.random.RandomState(RANDOM_STATE + i).choice(
        len(dataset), size=min(EVAL_SAMPLES, len(dataset)), replace=False
    ).tolist())
    for i in range(N_CALLS)
)
# Semplificazione: usa un nuovo seed per la validazione
val_rng = np.random.RandomState(RANDOM_STATE + 999)
val_indices = val_rng.choice(
    len(dataset),
    size=min(EVAL_SAMPLES, len(dataset)),
    replace=False
).tolist()

# Costruisci e valuta la configurazione ottimale
optimal_config = dict(BASE_RESIDUAL_CONFIG)
optimal_config.update(best_params)

from residual_clap_utils import build_pca_loader, fit_pca

clap_optimal = build_clap(
    model_type='residual',
    residual_config=optimal_config,
    version=CLAP_VERSION,
    use_cuda=torch.cuda.is_available()
)

pca_loader = build_pca_loader(
    wrapper=clap_optimal,
    dataset=dataset,
    max_samples=PCA_SAMPLES,
)
fit_pca(clap_optimal, pca_loader, max_samples=PCA_SAMPLES)

y_preds_opt, y_labels_val = evaluate_model(
    wrapper=clap_optimal,
    dataset=dataset,
    text_labels=text_labels,
    indices=val_indices,
    desc="Validazione config ottimale",
)
val_acc = accuracy(y_preds_opt, y_labels_val)

del clap_optimal
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n  Baseline accuracy (validazione):  {baseline_acc:.4f}")
print(f"  Optimal accuracy (validazione):   {val_acc:.4f}")
print(f"  Delta:                            {val_acc - baseline_acc:+.4f}")

In [ ]:
# ============================================================
# SALVATAGGIO RISULTATI FINALI
# ============================================================

final_report = {
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
    'clap_version': CLAP_VERSION,
    'optimization_params': {
        'n_calls': N_CALLS,
        'n_initial_points': N_INITIAL_POINTS,
        'eval_samples': EVAL_SAMPLES,
        'pca_samples': PCA_SAMPLES,
    },
    'param_space': {
        'n_components_ratio': list(PARAM_SPACE['n_components_ratio']),
        'target_layers_options': PARAM_SPACE['target_layers_options'],
    },
    'baseline_accuracy': float(baseline_acc),
    'best_params': best_params,
    'best_score_during_opt': float(best_score),
    'validation_accuracy': float(val_acc),
    'delta_validation': float(val_acc - baseline_acc),
    'all_trials': [
        {
            'trial': r['trial'],
            'n_components_ratio': float(r['n_components_ratio']),
            'target_layers': r['target_layers'],
            'accuracy': float(r['accuracy']),
            'delta_vs_baseline': float(r['delta_vs_baseline']),
        }
        for r in all_results
    ],
    'elapsed_minutes': elapsed,
}

with open('/kaggle/working/hp_tuning_results.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print("\n✅ Report completo salvato in hp_tuning_results.json")
print("\nFile generati:")
for f in Path('/kaggle/working').glob('hp_*'):
    print(f"  {f.name}")

In [ ]:
# ============================================================
# RIEPILOGO FINALE
# ============================================================

print(f"\n{'='*60}")
print(f"{'RIEPILOGO FINALE':^60}")
print(f"{'='*60}")
print(f"  Dataset:             {DATASET_NAME}")
print(f"  Trial eseguiti:      {N_CALLS}")
print(f"  Baseline accuracy:   {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print(f"")
print(f"  ★ Configurazione ottimale trovata:")
print(f"    n_components_ratio = {best_params['n_components_ratio']:.4f}")
print(f"    target_layers      = {best_params['target_layers']}")
print(f"")
print(f"  Best score (opt):    {best_score:.4f} ({best_score*100:.1f}%)")
print(f"  Validation accuracy: {val_acc:.4f} ({val_acc*100:.1f}%)")
print(f"  Delta (val):         {val_acc - baseline_acc:+.4f}")
print(f"  Tempo totale:        {elapsed:.1f} minuti")
print(f"{'='*60}")